# 📗 GDS 시작하기: 그래프 투영과 차수

31일차의 질문은 **"이 약과 비슷한 약은?"** 이었습니다. 기준 약(`Simvastatin`)을 정하고, 유전자를 함께 가리키는 약을 **두 홉 안에서** 세면 답이 나왔습니다. Cypher 패턴 하나로 충분했습니다.

오늘의 질문은 다릅니다.

> **"이 그래프에서 가장 중요한 노드는 무엇인가?"**

기준 노드가 없습니다. 노드 15,540개 **전부에 점수를 매겨 순위를 내야** 하고, 이웃의 이웃을 그래프 끝까지 따라가며 **반복 계산**해야 합니다. 이런 **전역 계산**을 맡는 엔진이 <strong>GDS(Graph Data Science)</strong>입니다.

**오늘 할 일.** GDS 를 확인하고, 분석의 출발점인 <strong>그래프 투영(projection)</strong>을 배웁니다. 6절에서 첫 계산으로 **차수(degree, 선의 개수)** 를 돌려 봅니다. 선의 개수만이 아니라 **이웃이 얼마나 중요한지까지 반영하는** PageRank 같은 중심성은 다음 시간(교안_02)에 배웁니다.

**데이터.** 시연은 **Hetionet** 의료 지식 그래프(노드 15,540개·관계 91,966개, 약물·질병·유전자·증상·약효분류)로, **따라하기는 수도권 전철 그래프**로 합니다. 도메인이 다른 그래프에 같은 절차를 대 봐야, 배운 것이 어디서나 통하는 절차인지 확인됩니다.

## 투영은 어디에 쓰일까요?

실무의 그래프 분석은 **원본 DB 에서 직접 돌리지 않습니다.** 분석할 조각만 메모리에 떠서(투영) 그 위에서 계산합니다. 원본은 서비스가 쓰고 있어 건드리면 안 되고, 반복 계산은 메모리에서 해야 빠르고, **질문마다 담을 것이 다르기** 때문입니다.

| 현장의 질문 | 무엇을 담나 |
|---|---|
| "이 고객이 다음에 살 상품은" | 고객·상품 노드와 구매 관계만 |
| "이 거래망에 돈세탁 고리가 있나" | 계좌 노드와 송금 관계만, **방향을 살려서** |
| "이 질문과 관련 깊은 개체는"(35일차의 GraphRAG) | 지식 그래프에서 질문에 걸린 **관계 종류만** |

같은 원본에서 **질문마다 다른 투영**을 만듭니다. 그래서 "무엇을 담을지 고르는 일"이 곧 분석의 첫 단계이고, 오늘 배우는 것이 그것입니다.

## ⏪ 복습: 앞 일차에서 배운 것

- **Python Driver**: `.env` 로 Neo4j 에 연결하고, 이 수업의 `run_cypher("쿼리", 파라미터=값)` 헬퍼로 Cypher 를 실행해 결과를 dict 리스트로 받았습니다.
- **적재·조회**: CSV 를 읽어 노드·관계를 적재하고, `MATCH`·`WHERE`·집계로 조회했습니다. 적재할 때 **레이블을 찍어 인덱스를 타게** 하는 것이 속도를 갈랐습니다.
- **31일차 추천 랭킹**: 가운데 노드를 함께 가리키는 것을 세어 순위를 냈습니다. 다만 **기준 노드를 지정해야** 했고, 원점수(공유 개수)와 비율(공유/전체)이 서로 다른 답을 준다는 것을 확인한 뒤 **비율에 최소 조건을 붙이는 선**에서 마무리했습니다. 그 손질을 알고리즘이 대신하는 노드 유사도는 34일차에서 다룹니다. 오늘은 그 앞 단계인 **전역 계산과 투영**을 배웁니다.
- 오늘은 이 위에 <strong>그래프 전체를 계산하는 분석 엔진(GDS)</strong>을 얹습니다. 1-2 에서 설치를 확인하고 2-1 에서 첫 투영을 만듭니다.

**오늘의 목표**

**1. GDS 엔진**
- [ ] (1-1) **GDS** 가 무엇이고 왜 별도 엔진인지 설명한다.
- [ ] (1-2) `gds.version()` 으로 설치를 확인한다.

**2. 그래프 투영**
- [ ] (2-1) **그래프 투영**(`gds.graph.project`)이 왜 필요한지를 **메모리 사본** 개념으로 이해한다.

**3. 방향과 검산**
- [ ] (3-1) 관계의 **방향을 지운 투영**(`orientation`)을 만들고, 제대로 만들어졌는지 **검산**한다.
- [ ] (3-2) 검산이 어긋났을 때 **`schemaWithOrientation`** 으로 어느 관계가 범인인지 짚는다.

**4. 투영 관리**
- [ ] (4-1) 투영을 **목록 조회**(`gds.graph.list`)한다.
- [ ] (4-2) 투영이 쓰는 **메모리를 재고**(`memoryUsage`), 만들기 전에 `estimate` 로 가늠하고, 다 쓴 투영을 **삭제**(`gds.graph.drop`)한다.

**5. Cypher 투영**
- [ ] (5-1) **Cypher 투영**(집계 함수형)으로 조건을 걸어 담고, 원본에 없는 관계도 만들어 본다.

**6. 차수**
- [ ] (6-1) **차수(degree)** 를 `gds.degree.stream` 으로 재고 순위로 읽는다.
- [ ] (6-2) `orientation` 으로 **나가는·들어오는·양쪽** 중 무엇을 셀지 고른다.
- [ ] (6-3) `relationshipTypes` 로 **어떤 관계만 셀지** 고른다.

아래 준비 셀들을 위에서부터 실행하세요. **연결 → 초기화(투영·그래프) → 데이터 적재** 순서입니다. 반드시 **실습 전용 DB**(`.env` 의 전용 접속 정보)에 연결하세요. 초기화 셀이 그래프를 모두 지웁니다.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
UNIT_LABELS = ['Compound', 'DayType', 'Disease', 'ExDenseEvent', 'ExDisease', 'ExDomestic', 'ExDrug', 'ExEvent', 'ExForeign', 'ExIdKey', 'ExKing', 'ExKinng', 'ExNameKey', 'ExNodeKeyDemo', 'ExPerson', 'ExReign', 'ExScopeEvent', 'ExThrone', 'ExUniqueDemo', 'ExWorld', 'ExYear', 'FlatDrug', 'FlatKing', 'FlatOrder', 'Gene', 'IdKey', 'Line', 'NameKey', 'NodeClass', 'NodeDay', 'NodeDrug', 'NodeKing', 'NodeMonth', 'NodeOrder', 'NodeYear', 'PharmacologicClass', 'Station', 'SurveyDay', 'SurveyYear', 'Symptom', 'TempStation', 'TryDay', 'TryLine', 'TryStation']   # 이 단원이 만드는 레이블 전부(앞 일차가 남긴 것까지)

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# all 로 보는 이유: any 로 보면 :Person:PatientRecord 처럼 한 레이블만 겹치는 남의 노드가 통과합니다
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=UNIT_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 가 실습 전용 DB 를 가리키는지 먼저 확인하세요.\n"
        "주소가 맞다면 위 레이블은 앞 실습이 남긴 것입니다. UNIT_LABELS 에 더하고 다시 실행하세요.")

# 여기까지 왔으면 이 DB 에는 이 단원이 만든 노드밖에 없습니다.
# 1) 메모리에 올라온 투영부터 내린다. 투영은 이름이 겹치면 다시 못 만들어 재실행이 막힌다
for _g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($name) YIELD graphName RETURN graphName",
               name=_g["graphName"])

# 2) 저장된 그래프를 지운다. DETACH: 노드에 붙은 관계까지 함께 지운다
run_cypher("MATCH (n) DETACH DELETE n")

print("초기화 완료:", NEO4J_URI,
      "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"],
      "· 남은 투영:", len(run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName")))

In [ ]:
# [제공 코드] 의료 지식 그래프 적재: 이 셀은 실행만 하세요(2초쯤 걸립니다).
# data/hetionet_*.csv 는 Hetionet v1.0 에서 재배포 가능한 출처(CC0/CC BY)만 골라 낸 조각입니다.
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")   # 정답 폴더에서도 돌게
NODE_LABELS = ["Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"]
# 관계 타입마다 양끝 레이블이 정해져 있습니다. 적재할 때 이 표로 레이블을 찍어 줘야
# MATCH 가 인덱스를 타고, 그래야 9만 건이 몇 초 안에 들어갑니다.
REL_ENDS = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "RESEMBLES_CC": ("Compound", "Compound"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

# 1) 레이블마다 id 인덱스를 먼저 만든다. 관계를 붙일 때 이 인덱스로 노드를 찾는다
for _label in NODE_LABELS:
    run_cypher(f"CREATE INDEX {_label.lower()}_id IF NOT EXISTS FOR (n:{_label}) ON (n.id)")

# 2) 노드 csv 를 읽어 레이블별로 나눠 담는다. 레이블마다 CREATE 쿼리가 달라서 미리 갈라 둔다
_nodes = {_label: [] for _label in NODE_LABELS}
for _row in pd.read_csv(DATA_DIR / "hetionet_nodes.csv").to_dict("records"):
    _nodes[_row["label"]].append({"id": _row["id"], "name": _row["name"]})
for _label, _rows in _nodes.items():
    # 초기화 직후라 같은 노드가 있을 수 없다. MERGE 대신 CREATE 가 훨씬 빠르다
    run_cypher(f"UNWIND $rows AS row CREATE (n:{_label}) SET n.id = row.id, n.name = row.name",
               rows=_rows)

# 3) 관계 csv 도 타입별로 나눠 담는다
_edges = {_rel: [] for _rel in REL_ENDS}
for _row in pd.read_csv(DATA_DIR / "hetionet_edges.csv").to_dict("records"):
    _edges[_row["rel"]].append({"s": _row["source"], "t": _row["target"]})
for _rel, _rows in _edges.items():
    _src, _dst = REL_ENDS[_rel]   # 이 타입의 출발·도착 레이블을 위 표에서 꺼낸다
    # 2만 건씩 끊어 보낸다. 9만 건을 한 트랜잭션에 넣으면 메모리가 크게 뛴다
    for _start in range(0, len(_rows), 20000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{_src} {{id: row.s}}), (b:{_dst} {{id: row.t}}) "
                   f"CREATE (a)-[:{_rel}]->(b)", rows=_rows[_start:_start + 20000])

print("노드:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"],
      "/ 관계:", run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"])

## 데이터 살펴보기

분석을 시작하기 전에 **무엇이 들어 있는지** 먼저 봅니다. 아래 셀은 실행만 하세요.

| 노드 종류 | 뜻 |
|---|---|
| `Compound` | 약물 |
| `Disease` | 질병 |
| `Gene` | 유전자. 그 유전자가 만드는 **단백질까지 이 이름으로** 부릅니다 |
| `Symptom` | 증상 |
| `PharmacologicClass` | 약효분류(같은 작용을 하는 약들의 묶음) |

| 관계 타입 | 뜻 |
|---|---|
| `TREATS` | 약물이 질병을 **치료**한다(질병 자체를 조절) |
| `PALLIATES` | 약물이 질병의 증상을 **완화**한다 |
| `BINDS` | 약물이 유전자 산물(단백질)에 **결합**한다 |
| `UPREGULATES_CG` · `DOWNREGULATES_CG` | 약물이 유전자 발현을 올린다·내린다 |
| `ASSOCIATES` | 질병과 유전자의 **연관 보고** |
| `UPREGULATES_DG` · `DOWNREGULATES_DG` | 질병 상태에서 유전자 발현이 올라간다·내려간다 |
| `RESEMBLES_CC` · `RESEMBLES_DD` | 약물끼리·질병끼리 **닮음** |
| `PRESENTS` | 질병이 증상을 **보인다** |
| `INCLUDES` | 약효분류가 약물을 **포함**한다 |

> 이 데이터가 말하는 것은 **문헌에 보고된 연관**입니다. "이 약이 이 병에 듣는다"는 **효능 입증**과는 다릅니다. 오늘 계산하는 순위도 "이 그래프에서 중심에 있다"는 뜻이지 임상 근거가 아닙니다.

In [ ]:
# [제공 코드] 적재한 그래프를 훑어봅니다: 실행만 하세요.
# 1) 노드 종류(레이블)별 개수. 이 데이터는 노드마다 레이블이 하나뿐이라 labels(n)[0] 로 충분하다
for _row in run_cypher("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS cnt ORDER BY cnt DESC"):
    print(f"  {_row['label']:20} {_row['cnt']:>6}")

In [ ]:
# [제공 코드] (이어서)
# 2) 관계 종류(타입)별 개수. 어떤 축이 굵은지 여기서 눈에 들어온다
for _row in run_cypher("MATCH ()-[r]->() RETURN type(r) AS rel_type, count(*) AS cnt "
                       "ORDER BY cnt DESC"):
    print(f"  {_row['rel_type']:20} {_row['cnt']:>6}")

## 따라하기용 그래프: 수도권 전철

<img src="images/전철그래프_모델.png" width="860">

**따라하기만 도메인이 다른 그래프를 씁니다.** 시연과 같은 그래프의 다른 조각으로 시키면 방금 본 쿼리에서 레이블 이름만 바꿔 넣는 연습이 되기 쉽습니다. 도메인을 갈라 두면 "이 절차를 내 데이터에 어떻게 대나"를 매번 다시 판단하게 됩니다.

투영은 이 두 그래프 중 **원하는 쪽만 골라 담습니다.** 담을 것을 고르는 일이 곧 질문을 정하는 일이라는 오늘의 요점이 여기서 한 번 더 확인됩니다.

아래 셀은 실행만 하세요. 적재한 뒤에 **위의 훑어보기 셀을 다시 돌리면** `Station`·`Line` 과 `ON_LINE`·`NEXT_TO` 가 함께 나옵니다. 한 DB 에 그래프가 둘이기 때문입니다.

In [ ]:
# [제공 코드] 수도권 전철 그래프 적재: 이 셀은 실행만 하세요(1초쯤 걸립니다).
# data/seoul_subway_*.csv 는 30일차에서 쓴 것과 같은 파일입니다.
# 자료 출처: OpenStreetMap contributors (ODbL).
#   (:Station {name})-[:NEXT_TO {line, km}]->(:Station)   이웃한 두 역
#   (:Station)-[:ON_LINE]->(:Line {name})                 역이 속한 노선
# 앞의 의료 그래프와는 선이 하나도 이어지지 않습니다. 한 DB 에 별개의 그래프 둘이 있는 셈입니다.
import csv
from pathlib import Path

# 앞의 의료 적재 셀에서 이미 정했지만, 이 셀만 따로 돌려도 되도록 여기서 한 번 더 정한다
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")


def _read_subway(filename):
    """전철 CSV 한 장을 dict 목록으로 읽는다. 값은 전부 문자열이라 드라이버에 그대로 넘어간다."""
    with open(DATA_DIR / filename, encoding="utf-8") as f:
        return list(csv.DictReader(f))


_st = _read_subway("seoul_subway_stations.csv")

# 이름으로 찾을 일이 많으니 인덱스부터 만든다(적재 속도가 여기서 갈린다)
run_cypher("CREATE INDEX station_name IF NOT EXISTS FOR (n:Station) ON (n.name)")
run_cypher("CREATE INDEX line_name IF NOT EXISTS FOR (n:Line) ON (n.name)")

# 1) 역 노드. 초기화 직후라 같은 이름이 있을 수 없어 MERGE 대신 CREATE 가 빠르다
run_cypher("UNWIND $rows AS row CREATE (:Station {name: row.name})",
           rows=[{"name": _r["name"]} for _r in _st])

# 2) 노선 노드. 역마다 적힌 노선 이름을 모아 중복을 없앤다
_lines = sorted({_ln for _r in _st for _ln in _r["lines"].split("|")})
run_cypher("UNWIND $rows AS name CREATE (:Line {name: name})", rows=_lines)

# 3) 역-노선 소속. 한 역이 여러 노선에 속하면 그만큼 선이 생긴다(그게 환승역이다)
run_cypher("UNWIND $rows AS row "
           "MATCH (s:Station {name: row.s}), (l:Line {name: row.l}) "
           "CREATE (s)-[:ON_LINE]->(l)",
           rows=[{"s": _r["name"], "l": _ln} for _r in _st for _ln in _r["lines"].split("|")])

# 4) 이웃 구간. 두 노선이 같은 구간을 함께 지나는 곳이 있어 그때는 선이 두 번 생긴다
run_cypher("UNWIND $rows AS row "
           "MATCH (a:Station {name: row.f}), (b:Station {name: row.t}) "
           "CREATE (a)-[:NEXT_TO {line: row.line, km: toFloat(row.km)}]->(b)",
           rows=[{"f": _r["from"], "t": _r["to"], "line": _r["line"], "km": _r["km"]}
                 for _r in _read_subway("seoul_subway_edges.csv")])

print("역:", run_cypher("MATCH (n:Station) RETURN count(n) AS c")[0]["c"],
      "/ 노선:", run_cypher("MATCH (n:Line) RETURN count(n) AS c")[0]["c"],
      "/ 구간:", run_cypher("MATCH ()-[r:NEXT_TO]->() RETURN count(r) AS c")[0]["c"],
      "/ 소속:", run_cypher("MATCH ()-[r:ON_LINE]->() RETURN count(r) AS c")[0]["c"])

---
# 1. GDS: 그래프를 계산하는 엔진

여기서는 31일차의 집계 랭킹과 무엇이 다른지를 짚어 GDS 가 어떤 계산을 맡는 엔진인지 잡습니다. 플러그인 설치 절차와 `gds.version()` 으로 켜졌는지 확인하는 법까지 한자리에서 봅니다.

## 1-1. GDS 는 무엇을 맡는 엔진인가

### 31일차의 추천 랭킹과 무엇이 다른가
31일차 3절에서 "이 약과 비슷한 약"을 이렇게 뽑았습니다.

```cypher
MATCH (c:Compound {name: 'Simvastatin'})-[:BINDS]->(g:Gene)<-[:BINDS]-(other:Compound)
WHERE other <> c
RETURN other.name AS 약물, count(DISTINCT g) AS 공유표적수
ORDER BY 공유표적수 DESC, 약물 LIMIT 5
```

계열 정보를 준 적이 없는데도 `-statin` 계열이 상위에 올라온, 목적을 충분히 달성한 쿼리입니다. 다만 이 방식으로는 다룰 수 없는 것이 셋 있습니다.

| | 31일차 집계 랭킹 | 오늘부터의 전역 계산 |
|---|---|---|
| **기준** | 기준 노드를 지정해야 한다(`{name: 'Simvastatin'}`). 지정하지 않으면 패턴이 성립하지 않는다 | 기준 없이 **모든 노드에 빠짐없이** 점수를 매긴다 |
| **범위** | 가운데 노드를 세는 것은 정확히 **2홉**이다. 세 홉부터는 계산에 들어오지 않는다 | 점수가 **그래프 끝까지** 전파된다 |
| **횟수** | Cypher 는 **한 번의 평가로 결과를 내는** 선언형 언어다 | 값이 **수렴할 때까지 반복**해 계산한다 |

<img src="images/집계랭킹_대_전역계산.png" width="900">

왼쪽이 31일차입니다. 기준 노드 하나를 정하고 그 둘레만 계산에 넣습니다. 나머지 그래프는 쿼리에 포함되지 않습니다. 오른쪽이 오늘부터입니다. 그래프 전체가 한 번에 계산에 들어가고, 점수가 선을 따라 전파되며 반복되어 노드마다 크기(점수)가 갈립니다.

### 왜 한 번에 풀리지 않고 반복이 필요한가
"중요한 이웃과 이어져 있으면 그 노드도 중요하다"고 정의하면 식이 **자기 자신을 참조**하게 됩니다. A 의 점수를 구하려면 B 의 점수가 필요하고, B 의 점수를 구하려면 다시 A 의 점수가 필요합니다. 이런 정의는 한 번의 계산으로 풀리지 않습니다. **초기값을 주고, 값이 더 변하지 않을 때까지 반복**해야 답이 정해집니다.

Cypher 에는 이 반복을 표현할 수단이 없습니다. `MATCH` 는 그래프를 한 번 훑어 결과를 냅니다. 그래서 **반복 계산을 대신 수행할 엔진**이 필요합니다. 그것이 **GDS** 입니다. 중심성·경로·군집 탐지 같은 **그래프 전역 계산**이 여기에 모여 있습니다.

## 1-2. 설치 확인: `gds.version()`

### 설치: Neo4j Desktop 플러그인
GDS 는 별도 **플러그인**입니다. Neo4j Desktop 에서는 클릭 한 번으로 켭니다.

- Neo4j Desktop 을 열고 실습용 **DBMS** 를 고른 뒤 오른쪽 **Plugins** 탭을 엽니다.
- **Graph Data Science Library** 항목의 **Install** 을 누르고, DBMS 를 **재시작(Restart)** 합니다.
- (Aura 를 쓴다면 플러그인을 직접 설치하지 않습니다. 대신 **Aura Graph Analytics** 가 계산 전용 세션을 띄워 같은 알고리즘을 돌려 줍니다. 무료 등급에서도 과금 없이 열리고, 한 번에 세션 하나·메모리 2GB·최대 4시간입니다. 투영을 만들 때 세션이 함께 뜨고, 투영을 내리면 세션도 닫힙니다.)

설치가 됐는지는 코드로 **버전을 찍어** 확인합니다.

In [ ]:
# GDS 가 설치·동작하는지 버전으로 확인: 문자열이 찍히면 준비 완료
# MATCH 가 없는 쿼리다. 그래프는 한 건도 읽지 않고 함수 하나만 불러 값을 받는다
ver = run_cypher("RETURN gds.version() AS version")   # 결과는 행 하나짜리 리스트로 온다
print("GDS 버전:", ver[0]["version"])

> `gds.version()` 이 버전 문자열을 돌려주면 GDS 가 이 데이터베이스에서 동작한다는 뜻입니다. `Unknown function 'gds.version'` 같은 에러가 난다면 플러그인이 아직 안 켜진 것입니다. 설치 후 **재시작**했는지 확인하세요.

> **버전 숫자 모양은 신경 쓰지 마세요.** GDS 는 예전 `2.5.0` 같은 표기에서 `2026.06.0` 처럼 **연도.월** 표기로 바뀌었습니다. 어느 모양이든 문자열이 찍히면 정상입니다.

### 🖐️ 함께 따라하기: GDS 가 무엇을 제공하는지 세어 보기

`CALL gds.list()` 는 이 GDS 가 제공하는 **프로시저·함수 목록**을 돌려줍니다. `YIELD name` 으로 이름만 받아 1) 전체가 몇 개인지, 2) 이름에 `pageRank` 가 들어간 것이 몇 개인지 세어 출력하세요.

**확인 기준**: 전체 개수는 수백 개(버전마다 다름), `pageRank` 가 든 이름 중에 `gds.pageRank.stream` 과 `gds.pageRank.write` 가 보이면 맞습니다. 다음 시간에 쓸 알고리즘이 이미 이 목록 안에 들어 있다는 것을 눈으로 확인하는 것이 목적입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CALL gds.list() YIELD name RETURN name 으로 이름 목록을 받는다
# 2) 전체 개수를 출력한다
# 3) 이름에 'pageRank' 가 들어간 것만 골라 개수를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 31일차에는 "이 약과 비슷한 약"을 Cypher 패턴 하나로 뽑았습니다. "이 그래프에서 가장 중심인 질병"에는 왜 GDS 가 필요한가요?

<details><summary>정답 보기</summary>

31일차 쿼리는 **기준 약을 지정해야** 성립했고, 그 기준에서 **2홉**까지만 셌습니다. "가장 중심"은 기준으로 삼을 노드가 없어 **모든 노드를 한 번에** 줄 세워야 하고, 각 노드의 점수가 이웃의 점수에 의존하므로 **값이 수렴할 때까지 반복**해야 답이 정해집니다. `MATCH` 는 한 번의 평가로 끝나는 문법이라 이 반복을 표현할 수 없습니다. GDS 가 이 전역 계산을 담당합니다.

</details>

**2.** `gds.version()` 을 실행하는 목적은 무엇인가요?

<details><summary>정답 보기</summary>

GDS 플러그인이 이 데이터베이스에 **설치되어 동작하는지** 확인하기 위해서입니다.

</details>

---
# 2. 그래프 투영: 분석은 메모리 사본 위에서 한다

GDS 알고리즘은 **아무 데서나 바로 돌지 않습니다.** 먼저 분석할 조각을 골라 메모리에 사본을 만들어야 하고, 그 사본을 **투영(projection)** 이라고 합니다. 여기서는 투영이 무엇이고 언제 만드는지 보고, `gds.graph.project` 의 세 자리(이름·레이블·관계)를 채워 첫 투영을 만듭니다.

## 2-1. 투영을 만들어 메모리에 올리기

### 왜 사본을 만드나요?
GDS 알고리즘은 디스크에 저장된 그래프를 **직접 쓰지 않습니다.** 먼저 분석에 필요한 부분만 골라 **메모리에 사본을 만들고**, 그 위에서 계산합니다.

이유는 앞 절에서 본 **반복 계산**입니다. 다음 시간에 볼 PageRank 는 기본 설정만으로도 같은 그래프를 **스무 번** 훑습니다. 그때마다 디스크를 읽으면 계산하는 시간보다 읽는 시간이 더 듭니다. 그래서 **한 번 읽어 메모리에 펴 두고** 그 위에서만 계산합니다. 여기에 두 가지 이점이 더해집니다. 계산이 원본을 건드리지 않아 **격리**되고, 노드 15,540개를 다 계산할 필요가 없을 때 **필요한 종류만 담아** 범위를 좁힐 수 있습니다.

<img src="images/투영_시점사본.png" width="820">

### 언제 만드나요?
투영을 만드는 것은 "이제부터 이 조각으로 분석하겠다"는 선언입니다. 실무에서 만드는 자리는 대개 이 셋입니다.

| 언제 | 무엇을 담나 | 왜 그렇게 하나 |
|---|---|---|
| **순위를 매길 때** | 그 순위와 상관있는 레이블·관계만 | 담지 않은 것은 계산에 끼지 못한다. 담을 것을 고르는 일이 곧 **질문을 정하는 일**이다 |
| **한 그래프에 여러 질문을 던질 때** | 질문마다 다른 조각을 **따로** | "무엇이 중심인가"와 "이 약과 가까운 것은"은 담을 것이 다르다. 다음 시간에 투영 두 개를 나란히 놓고 쓴다 |
| **같은 조각에 알고리즘을 바꿔 가며 돌릴 때** | 한 번 담아 두고 알고리즘만 갈아 끼운다 | 투영 하나에 차수·PageRank·매개 중심성을 차례로 돌릴 수 있다. 매번 다시 담을 이유가 없다 |

반대로 **안 만들어도 되는 때**도 분명합니다. "고혈압을 치료하는 약을 세어 달라"처럼 패턴 하나로 끝나는 조회는 그냥 Cypher 로 합니다. 투영은 메모리를 차지하므로, 전역 계산을 돌릴 것이 아니면 만들 이유가 없습니다.

그리고 **다 쓰면 내립니다**(`gds.graph.drop`). 메모리를 잡고 있을 뿐 아니라, 투영은 **만든 시점의 사본**이라 그 뒤 원본이 바뀌어도 따라가지 않기 때문입니다. 최신 데이터로 다시 분석하려면 내리고 다시 만듭니다.

### 문법

```cypher
CALL gds.graph.project(투영이름, 노드레이블, 관계설정)
YIELD graphName, nodeCount, relationshipCount
```

| 자리 | 넣는 것 | 예 |
|---|---|---|
| 투영이름 | 메모리에 붙일 이름(문자열) | `'treatGraph'` |
| 노드레이블 | 담을 노드 종류. 하나면 문자열, 여럿이면 리스트 | `['Compound', 'Disease']` |
| 관계설정 | 담을 관계 타입. 리스트 또는 설정 맵 | `['TREATS']` |

**시연은 약물-질병 축**(`Compound`·`Disease`·`TREATS`)으로 합니다. "어떤 약이 어떤 병을 치료하는가"만 담은 조각입니다.

In [ ]:
# 약물-질병 축만 메모리에 올린다. "어떤 약이 어떤 병을 치료하는가" 하나만 담은 조각이다
# 이름을 붙여 두면 이후 알고리즘 호출이 전부 이 이름으로 이 사본을 가리킨다
res = run_cypher("""
    CALL gds.graph.project('treatGraph', ['Compound', 'Disease'],
                           // 이 타입의 양끝 레이블이 앞 리스트에 다 있어야 관계가 따라 담긴다
                           ['TREATS'])
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount   // YIELD 로 고른 것을 RETURN 으로 돌려받아 파이썬에서 쓴다
""")
print(res[0])

- `nodeCount` 가 **1,667** 입니다. 약물 1,531개 + 질병 136개죠. **TREATS 관계가 하나도 없는 약까지 전부 담깁니다.** 레이블을 담으면 그 레이블의 노드는 다 들어옵니다.
- `relationshipCount` 는 **755** 입니다. 원본의 TREATS 관계 수와 같습니다.

> **왜 `YIELD` 가 필요한가**: GDS 프로시저는 여러 값을 한꺼번에 돌려줍니다. `YIELD` 로 **쓸 것만 골라야** 합니다. 안 쓰는 값까지 받으면 버전에 따라 경고가 뜨기도 합니다.

### 🖐️ 함께 따라하기: 역과 노선을 투영하기

시연은 **의료 그래프**였습니다. 따라하기는 **수도권 전철 그래프**로 같은 일을 합니다.

`Station` 과 `Line` 노드, 그 사이의 `ON_LINE` 관계를 **`lineGraph`** 라는 이름으로 투영하고 `nodeCount`·`relationshipCount` 를 출력하세요.

**확인 기준**: 노드 684개(역 659 + 노선 25), 관계 802건이 나오면 맞습니다.

> **레이블 둘을 다 담아야 그 사이의 관계가 따라옵니다.** `Station` 만 담으면 `ON_LINE` 의 도착점인 `Line` 이 투영 밖이라 관계가 통째로 빠져 **0건**이 되고, `Line` 만 담아도 이번엔 출발점이 빠져 마찬가지로 0건입니다. 둘 다 **에러 없이** 만들어지므로 관계 수를 보지 않으면 알아채지 못합니다. 시연에서 본 "관계는 양끝이 모두 담겨야 따라온다"가 이 그래프에서도 그대로입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gds.graph.project 로 lineGraph 를 만든다
#    노드 레이블은 ['Station', 'Line'], 관계는 ['ON_LINE']
# 2) YIELD 로 nodeCount, relationshipCount 를 받아 출력한다

### ✅ 바로 확인 퀴즈

**1.** GDS 알고리즘이 저장된 그래프를 직접 쓰지 않고 **투영(메모리 사본)** 위에서 계산하는 이유를 하나 드세요.

<details><summary>정답 보기</summary>

메모리 전용 구조라 **반복 계산이 빠르고**, 원본을 건드리지 않아 **격리**되며, 필요한 부분만 골라 담을 수 있어 **계산 범위를 좁힐** 수 있습니다.

</details>

**2.** `treatGraph` 의 `nodeCount` 가 관계 수(755)보다 훨씬 큰 1,667인 이유는 무엇인가요?

<details><summary>정답 보기</summary>

레이블을 담으면 **그 레이블의 노드가 전부** 들어오기 때문입니다. `TREATS` 관계가 하나도 없는 약도 `Compound` 라는 이유로 투영에 담깁니다.

</details>

---
# 3. 방향을 지운 투영과 검산

여기서는 `orientation` 세 가지를 보고 관계를 무방향으로 담아 봅니다. 이어서 투영의 관계 수를 Cypher 로 센 건수와 맞대 봅니다. 노드마다 붙은 선의 개수(차수) 분포까지 견주면, 오늘 내내 쓸 **검산** 절차가 됩니다.

## 3-1. 방향을 지우고 관계 수로 검산하기

### 왜 방향을 지울까요?
원본의 `TREATS` 는 약물에서 질병으로 가는 **한 방향** 화살표입니다. 그런데 "이 약과 비슷한 약"이나 "두 무리를 잇는 길목"을 물을 때는 화살표 방향이 오히려 방해가 됩니다. 질병에서 약으로 거슬러 갈 수 없으면 **한 걸음 만에 길이 끊기기** 때문입니다.

그래서 투영을 만들 때 관계마다 **방향 처리**를 지정할 수 있습니다.

| `orientation` | 뜻 |
|---|---|
| `'NATURAL'` | 원본 방향 그대로(기본값) |
| `'UNDIRECTED'` | 양방향으로 담는다. 관계 하나가 **두 건**으로 들어간다 |
| `'REVERSE'` | 방향을 뒤집어 담는다 |

관계 자리에 리스트 대신 **설정 맵**을 넣으면 됩니다.

```cypher
CALL gds.graph.project('이름', ['A', 'B'], {TREATS: {orientation: 'UNDIRECTED'}})
```

In [ ]:
# 같은 약물-질병 축을 이번엔 방향 없이 담는다. 리스트가 아니라 설정 맵을 넘긴다
res = run_cypher("""
    CALL gds.graph.project('treatUndirected', ['Compound', 'Disease'],
                           // 맵의 키는 관계 타입 이름이다. 철자가 틀리면 에러가 난다(조용히 넘어가지 않는다)
                           {TREATS: {orientation: 'UNDIRECTED'}})
    YIELD nodeCount, relationshipCount   // 방향을 지워도 nodeCount 는 그대로다. 늘어나는 것은 관계뿐
    RETURN nodeCount, relationshipCount
""")[0]
print("무방향 투영:", res)

In [ ]:
# 원본에서 Cypher 로 직접 센 관계 수와 비교한다. 무방향이면 정확히 2배여야 한다
# 셀 때는 화살표를 살려 둔다. ()-[r:TREATS]-() 로 세면 같은 관계를 양끝에서 두 번 세어 비교가 망가진다
raw = run_cypher("MATCH ()-[r:TREATS]->() RETURN count(r) AS cnt")[0]["cnt"]
print("원본 TREATS 건수:", raw, "/ 2배인가?", res["relationshipCount"] == raw * 2)

### 투영을 만들면 반드시 검산합니다

무방향 투영은 관계 하나를 **양쪽 두 건**으로 담습니다. 그래서 검산이 간단합니다.

| 상태 | `relationshipCount` |
|---|---|
| 무방향으로 담겼다 | Cypher 로 센 건수의 **2배** |
| 방향이 남아 있다 | Cypher 로 센 건수와 **같음** |

**이 검산을 건너뛰면 조용히 틀립니다.** `orientation` 을 적었는데도 문법이 조금 어긋나 무시되면, 투영은 만들어지고 알고리즘도 돌지만 **답만 달라집니다.** 다음 시간 PageRank 에서 이 차이가 얼마나 크게 벌어지는지 직접 봅니다(방향을 남기면 약효분류 345개가 전부 같은 점수로 동점이 됩니다).

`gds.graph.list` 의 **`degreeDistribution`** 을 함께 보면 더 확실합니다. 노드가 평균 몇 개의 선을 가졌는지 보여 줍니다.

In [ ]:
# 방향이 있는 투영과 없는 투영의 차수 분포를 나란히 본다
for name in ["treatGraph", "treatUndirected"]:
    # degreeDistribution 은 min·max·mean·p50 같은 요약값을 담은 dict 로 온다
    # 무방향이면 mean 이 정확히 2배가 되고, p50(중앙값) 0 은 선 없는 노드가 절반을 넘는다는 뜻이다
    dist = run_cypher("""
        CALL gds.graph.list($n)   // 이름을 문자열로 이어 붙이지 않고 파라미터로 넘긴다
        YIELD degreeDistribution
        RETURN degreeDistribution
    """, n=name)[0]["degreeDistribution"]
    print(f"{name:16} mean={dist['mean']:.4f}  p50={dist['p50']}  max={dist['max']}")

- 무방향 쪽의 `mean` 이 정확히 2배입니다. 관계 하나를 양쪽에서 세기 때문입니다.
- `p50`(중앙값)이 **0** 인 것은 방향 때문이 아닙니다. **선이 하나도 없는 노드가 절반을 넘는다**는 뜻이죠. 약물 1,531개 중 치료 관계가 붙은 것은 일부뿐이니 당연한 결과입니다. 2절에서 본 "레이블을 담으면 그 노드가 전부 들어온다"가 여기서 숫자로 드러납니다.

## 3-2. 검산이 틀렸을 때 범인을 짚는 법: `schemaWithOrientation`

2배 검산은 **숫자 두 개를 견주는 총량 비교**입니다. 관계 타입이 하나면 틀린 곳이 곧 그 타입이라 이것으로 충분합니다. 그런데 타입이 여럿이면 "틀렸다"까지만 알 수 있고 **어느 타입인지는 못 짚습니다.** 다음 시간에 관계 타입 **12개**를 한꺼번에 담는데, 거기서 검산이 `False` 를 주면 후보가 12개입니다.

`gds.graph.list` 의 **`schemaWithOrientation`** 이 그 12개를 한 번에 펼쳐 줍니다.

```cypher
CALL gds.graph.list('투영이름') YIELD schemaWithOrientation
```

In [ ]:
# 일부러 한 번 틀려 본다: 타입 둘 중 PALLIATES 에만 orientation 을 빠뜨린다
run_cypher("CALL gds.graph.drop('mixedGraph', false) YIELD graphName RETURN graphName")
res = run_cypher("""
    CALL gds.graph.project('mixedGraph', ['Compound', 'Disease'],
        {TREATS: {orientation: 'UNDIRECTED'}, PALLIATES: {}})   // PALLIATES 만 설정을 비웠다
    YIELD relationshipCount
    RETURN relationshipCount
""")[0]["relationshipCount"]
raw = run_cypher("MATCH ()-[r:TREATS|PALLIATES]->() RETURN count(r) AS cnt")[0]["cnt"]
print(f"투영 {res} / 원본 {raw} / 2배인가? {res == raw * 2}")

In [ ]:
# 검산은 '틀렸다' 까지만 말한다. 범인은 스키마를 펼쳐야 보인다
schema = run_cypher("CALL gds.graph.list('mixedGraph') "
                    "YIELD schemaWithOrientation RETURN schemaWithOrientation"
                    )[0]["schemaWithOrientation"]
for rel_type, spec in schema["relationships"].items():
    print(f"  {rel_type:12} {spec['direction']}")

In [ ]:
run_cypher("CALL gds.graph.drop('mixedGraph') YIELD graphName RETURN graphName")

`PALLIATES` 만 `DIRECTED` 로 찍힙니다. **범인을 이름으로 지목해 줍니다.**

이 한 줄은 "지금 이 투영에 정확히 무엇이 담겨 있나"를 보는 데도 씁니다. 관계의 방향뿐 아니라 노드에 붙은 속성까지 함께 나오기 때문입니다(다음 시간에 계산 결과를 투영에 남기고 나서 이 자리로 확인합니다).

> 옛 이름 `schema` 도 있지만 이 버전에서는 **deprecated 경고가 뜨고 방향을 안 보여 줍니다.** `schemaWithOrientation` 을 쓰세요.

### 🖐️ 함께 따라하기: 역-노선을 무방향으로 담고 검산하기

`Station` 과 `Line` 을 담고 `ON_LINE` 을 **무방향으로** `lineUndirected` 에 투영하세요. 그리고 Cypher 로 센 원본 `ON_LINE` 건수와 비교해 **2배가 맞는지** 직접 확인해 출력하세요.

**확인 기준**: 원본 802건, 투영 1,604건, 비교 결과 `True` 가 나오면 제대로 담긴 것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gds.graph.project 로 lineUndirected 를 만든다
#    레이블 ['Station', 'Line'], 관계는 {ON_LINE: {orientation: 'UNDIRECTED'}}
# 2) MATCH ()-[r:ON_LINE]->() RETURN count(r) 로 원본 건수를 센다
# 3) 투영의 relationshipCount 가 원본의 2배인지 비교해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 무방향으로 담은 투영의 `relationshipCount` 가 원본 건수의 2배인 이유는?

<details><summary>정답 보기</summary>

무방향은 관계 하나를 **양쪽 방향 두 건**으로 저장하기 때문입니다. `A-B` 를 `A->B` 와 `B->A` 로 함께 담습니다.

</details>

**2.** `orientation` 을 적었는데 투영의 관계 수가 원본과 같게 나왔습니다. 무엇을 의심해야 할까요?

<details><summary>정답 보기</summary>

설정이 적용되지 않고 **방향 투영이 만들어졌다**고 봐야 합니다. 흔한 원인은 둘입니다. 관계 자리에 **설정 맵 대신 리스트**를 넘겼거나, 관계 타입이 여럿인데 **일부에만 `orientation` 을 적은** 경우입니다.

> 타입 **이름을 틀리게 적은 경우는 여기에 해당하지 않습니다.** 그때는 조용히 넘어가지 않고 `Invalid relationship projection` 에러가 납니다. 에러가 나면 오히려 다행이고, **에러 없이 답만 달라지는 쪽**이 이 절이 경계하는 상황입니다.

</details>

---
# 4. 투영 관리: 목록 조회와 삭제

투영은 **메모리를 차지합니다.** 지금까지 만든 것이 몇 개나 올라와 있는지 보고, 다 쓴 것은 내려야 합니다.

| 하는 일 | 문법 |
|---|---|
| 전체 목록 | `CALL gds.graph.list() YIELD graphName, nodeCount, relationshipCount` |
| 하나만 조회 | `CALL gds.graph.list('이름') YIELD ...` |
| 삭제 | `CALL gds.graph.drop('이름') YIELD graphName` |
| 없어도 통과하는 삭제 | `CALL gds.graph.drop('이름', false) YIELD graphName` |
| 메모리 사용량 | `CALL gds.graph.list() YIELD graphName, nodeCount, memoryUsage` |
| 만들기 전 가늠 | `CALL gds.graph.project.estimate(레이블, 관계) YIELD requiredMemory` |

> `YIELD` 없이 `CALL gds.graph.drop('이름')` 만 쓰면 안 쓰는 컬럼까지 받게 되어 버전에 따라 **경고**가 뜹니다. 쓸 컬럼만 골라 받는 습관을 들이세요.

## 4-1. 목록 조회: 지금 무엇이 올라와 있나

In [ ]:
# 지금 메모리에 올라온 투영을 전부 나열한다. 여기 없는 이름은 메모리에 없는 것이다
rows = run_cypher("""
    CALL gds.graph.list()   // 인자 없이 부르면 전체, 이름을 하나 주면 그것만 돌려준다
    YIELD graphName, nodeCount, relationshipCount
    RETURN graphName, nodeCount, relationshipCount
    ORDER BY graphName   // 목록 순서는 보장되지 않는다. 실행할 때마다 뒤바뀌지 않게 이름으로 정렬한다
""")
for row in rows:
    print(f"  {row['graphName']:20} 노드 {row['nodeCount']:>6,}  관계 {row['relationshipCount']:>6,}")

## 4-2. 얼마나 차지하는지 재기

"메모리를 차지한다"고 했으니 **얼마나 차지하는지** 볼 수 있어야 합니다. `gds.graph.list()` 는 `memoryUsage` 도 함께 돌려줍니다. 아직 만들지 않은 투영은 `gds.graph.project.estimate` 로 **만들기 전에** 가늠합니다. 큰 그래프를 담다가 메모리가 모자라는 사고는 이 한 줄로 미리 걸러집니다.

In [ ]:
# 투영이 메모리를 얼마나 쓰는지 본다. 노드 수만이 아니라 관계 수와 담는 방식이 함께 영향을 준다
# memoryUsage 는 '123 KiB' 같은 문자열로 온다. 눈으로 읽는 값이지 더하거나 비교할 수는 없다
for row in run_cypher("""
    CALL gds.graph.list()
    YIELD graphName, nodeCount, memoryUsage   // 앞 셀과 같은 프로시저다. 꺼내는 컬럼만 바꿨다
    RETURN graphName, nodeCount, memoryUsage
    ORDER BY graphName
"""):
    print(f"  {row['graphName']:20} 노드 {row['nodeCount']:>6,}  메모리 {row['memoryUsage']}")


In [ ]:
# 아직 만들지 않은 투영은 estimate 로 미리 재 볼 수 있다. 실제로 담지는 않는다
# 돌려주는 값이 하나가 아니라 '최소 ... 최대' 범위인 것은, 담기는 구조에 따라 폭이 생기기 때문이다
est = run_cypher("""
    CALL gds.graph.project.estimate(['Compound', 'Disease'], ['TREATS'])
    // project 와 인자가 같은데 이름 자리만 없다. 만들지 않으니 붙일 이름도 필요 없다
    YIELD requiredMemory
    RETURN requiredMemory
""")
print("treatGraph 를 만들 때 드는 메모리(예상):", est[0]["requiredMemory"])

### 일부러 한 번 틀려 보기: 같은 이름으로 또 만들면?

투영 이름은 **중복될 수 없습니다.** 이미 있는 이름으로 다시 만들면 에러가 납니다. 노트북을 다시 실행할 때 자주 만나는 에러라 미리 눈으로 봐 둡니다.

In [ ]:
# 이미 있는 이름으로 다시 투영해 본다. 에러 메시지를 눈으로 익히는 것이 목적이다
try:
    run_cypher("""
        CALL gds.graph.project('treatGraph', ['Compound', 'Disease'], ['TREATS'])
        // 2절에서 쓴 것과 한 글자도 다르지 않다. 이름이 이미 쓰이고 있다는 이유만으로 막힌다
        YIELD graphName
        RETURN graphName
    """)
except Exception as error:
    # 메시지가 길지만 핵심은 한 마디다. 그 문장이 시작하는 위치만 잘라서 본다
    text = str(error)
    marker = text.find("A graph with name")
    print("에러:", text[marker:marker + 60])

> 긴 메시지 안의 **`A graph with name 'treatGraph' already exists.`** 가 핵심입니다. 이 에러를 만나면 만들기 전에 같은 이름을 먼저 지우면 됩니다.

In [ ]:
# 1) 다 쓴 투영을 내린다. 이름을 그대로 부르면 그 하나가 메모리에서 사라진다
run_cypher("CALL gds.graph.drop('treatUndirected') YIELD graphName RETURN graphName")
# 2) 없는 이름을 지워 본다. 두 번째 인자 false 를 주면 그 이름이 없어도 에러가 나지 않는다
run_cypher("CALL gds.graph.drop('없는이름', false) YIELD graphName RETURN graphName")   # 지울 것이 없어 빈 결과
# 3) 목록을 다시 뽑아 정말 빠졌는지 눈으로 본다. 내렸다고 믿지 말고 남은 것을 센다
left = run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName ORDER BY graphName")
print("남은 투영:", [row["graphName"] for row in left])

### 🖐️ 함께 따라하기: 따라하기용 투영 정리하기

앞에서 만든 `lineUndirected` 를 내리고, 남은 투영 이름을 출력하세요. 그다음 **이미 지운 이름**을 `drop(이름, false)` 로 한 번 더 지워 보고 에러가 나지 않는 것을 확인하세요.

**확인 기준**: 목록에서 `lineUndirected` 가 사라지고, 두 번째 삭제는 조용히 넘어갑니다. 2절에서 만든 `lineGraph` 는 **6절 따라하기에서 다시 쓰고 5절에서 크기를 견줄 때도 필요하므로** 남겨 둡니다. 내릴 것은 `lineUndirected` 뿐입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gds.graph.drop('lineUndirected') 를 YIELD graphName 과 함께 실행한다
# 2) gds.graph.list() 로 남은 이름을 뽑아 출력한다
# 3) 방금 지운 이름을 drop(이름, false) 로 한 번 더 부르고, 에러가 안 나는지 확인한다

### ✅ 바로 확인 퀴즈

**1.** 노트북을 다시 실행했더니 `A graph with name 'treatGraph' already exists.` 에러가 났습니다. 어떻게 해결하나요?

<details><summary>정답 보기</summary>

만들기 전에 같은 이름을 먼저 지우면 됩니다. `CALL gds.graph.drop('treatGraph', false) YIELD graphName` 처럼 **없어도 통과하는 삭제**를 앞에 두는 것이 안전합니다.

</details>

**2.** `gds.graph.drop('이름')` 과 `gds.graph.drop('이름', false)` 의 차이는?

<details><summary>정답 보기</summary>

두 번째 인자는 "없으면 에러를 내라"는 뜻입니다. 기본값 `true` 는 그 이름이 없으면 에러를 내고, `false` 를 주면 **조용히 넘어갑니다.**

</details>

---
# 5. 조건을 걸어 담기: Cypher 투영

여기서는 집계 함수형 `gds.graph.project` 로 `MATCH`·`WHERE` 가 고른 짝만 담아 봅니다. 원본에 없는 관계를 만들어 담는 데까지 가 봅니다. 그 전에 이름을 하나 정해 둡니다. 지금까지 쓴 `CALL gds.graph.project(이름, 레이블, 관계)` 방식을 **native(기본형) 투영**이라 부릅니다. 레이블과 타입 이름만 주면 GDS 가 통째로 담아 주는 방식이죠. 마지막에 이 native 방식과 무엇이 다른지 표로 정리합니다.

## 5-1. `MATCH` 로 고른 짝만 담기

### 왜 필요할까요?
지금까지 쓴 투영은 **레이블과 관계 타입을 통째로** 담습니다. "`Compound` 를 담아라"고 하면 약물 1,531개가 다 들어옵니다. 그런데 실무 질문은 대개 조각을 요구합니다.

> "**고혈압·천식·편두통** 이 세 병과 그 치료제만 놓고 보면 어떤가?"

native 투영으로는 이 조각을 만들 수 없습니다. `Disease` 를 담는 순간 질병 136개가 다 따라옵니다. 이럴 때 쓰는 것이 **Cypher 투영**입니다. `MATCH`·`WHERE` 로 고른 노드 짝만 담습니다.

### 문법

지금까지의 `CALL gds.graph.project(...)` 는 **프로시저**였습니다. Cypher 투영은 이름은 같지만 **집계 함수**라 `RETURN` 자리에 씁니다. 레이블 대신 **출발 노드와 도착 노드를 직접 넘기는** 것이 다릅니다.

```cypher
MATCH (a)-[:관계]->(b)
WHERE 조건
RETURN gds.graph.project('투영이름', a, b,
       {sourceNodeLabels: labels(a), targetNodeLabels: labels(b)}) AS g
```

> **옛 문법 주의**: 인터넷 자료에는 `CALL gds.graph.project.cypher('이름', 노드쿼리, 관계쿼리)` 라는 **프로시저**가 자주 나옵니다. 지금 버전에서는 **deprecated** 이고 위의 집계 함수형이 그 자리를 대신합니다. 새로 쓰는 코드에는 함수형을 쓰세요.

In [ ]:
# 질병 셋과 그 치료제만 담은 조각을 만든다. native 투영으로는 만들 수 없는 모양이다
# 네 번째 인자로 레이블을 함께 넘겨야 투영 안에서도 노드 종류를 구분할 수 있다
res = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WHERE d.name IN ['hypertension', 'asthma', 'migraine']   // 이 조건이 곧 투영의 크기다
    // CALL 이 아니라 RETURN 이다. 집계 함수라 행마다 불려 짝을 하나씩 쌓아 간다
    RETURN gds.graph.project('narrowGraph', c, d,
           {sourceNodeLabels: labels(c), targetNodeLabels: labels(d)}) AS g
""")[0]["g"]
print(f"{res['graphName']}: 노드 {res['nodeCount']}, 관계 {res['relationshipCount']}")


In [ ]:
# 같은 TREATS 인데 담는 규칙이 다르면 크기가 얼마나 달라지는지 나란히 본다
for name in ["treatGraph", "narrowGraph"]:
    row = run_cypher("""
        CALL gds.graph.list($n)   // 담은 방식이 native 든 Cypher 든 만들어진 뒤로는 똑같이 다룬다
        YIELD nodeCount, relationshipCount
        RETURN nodeCount, relationshipCount
    """, n=name)[0]
    print(f"  {name:12} 노드 {row['nodeCount']:>5,}  관계 {row['relationshipCount']:>4,}")


- native `treatGraph` 는 노드 **1,667개**를 담았습니다. `TREATS` 가 하나도 없는 약까지 레이블만 보고 다 들어온 결과입니다.
- Cypher `narrowGraph` 는 노드 **112개**뿐입니다. 질병 3개와 그 치료제 109개죠. 관계도 **112건**으로, 그 세 병에 붙은 `TREATS` 만 담겼습니다.

같은 데이터, 같은 관계 타입인데 **담는 규칙이 달라 노드 수가 열다섯 배 가까이** 벌어졌습니다.

### 원본에 없는 관계도 만들 수 있습니다

Cypher 투영에 넘기는 두 노드는 **원본에서 직접 이어져 있지 않아도 됩니다.** `MATCH` 가 맺어 준 짝이면 그대로 관계가 됩니다. "같은 약효분류에 든 약끼리 이어 보자"처럼 **원본에 없는 관계**를 만들어 놓고 그 위에서 분석할 수 있습니다.

In [ ]:
# 같은 약효분류(PharmacologicClass)에 든 약 두 개를 곧바로 이어 담는다
# 원본에는 이 관계가 없다. 약효분류를 거쳐 두 걸음 걸어야 나오는 짝을 한 걸음으로 만드는 것이다
res = run_cypher("""
    MATCH (p:PharmacologicClass)-[:INCLUDES]->(c1:Compound),
          (p)-[:INCLUDES]->(c2:Compound)
    // p 는 짝을 맺는 데만 쓰고 투영에는 담기지 않는다. 담기는 것은 c1, c2 와 그 사이의 선뿐이다
    WHERE elementId(c1) < elementId(c2)   // 같은 짝이 (a,b)와 (b,a)로 두 번 담기는 것을 막는다. 두 약이 약효분류를 둘 공유하면 그만큼 선이 겹쳐 담긴다
    RETURN gds.graph.project('sameClassGraph', c1, c2,
           {sourceNodeLabels: labels(c1), targetNodeLabels: labels(c2)}) AS g
""")[0]["g"]
print(f"{res['graphName']}: 노드 {res['nodeCount']}, 관계 {res['relationshipCount']}")


약 621개가 3,156개의 선으로 이어졌습니다. 이 선은 **원본 데이터베이스 어디에도 없습니다.** "약효분류를 공유하는 약들 중에서는 무엇이 중심인가" 같은 질문이 이제 한 번의 계산으로 풀립니다.

### 두 방식 정리

| 방식 | 담기는 규칙 | 언제 쓰나 |
|---|---|---|
| native `CALL gds.graph.project(이름, 레이블, 관계)` | 그 레이블·타입을 **전부** 담는다. 빠른 대신 조건을 못 건다 | 그래프 전체를 한 잣대로 잴 때 |
| Cypher `RETURN gds.graph.project(이름, 출발, 도착, 설정)` | `MATCH`·`WHERE` 로 **고른 짝만** 담는다. 원본에 없는 관계도 만든다 | 조건을 건 조각을 볼 때 |

방금 만든 셋을 실측값으로 견주면 이렇습니다.

| 투영 | 담은 방식 | 노드 | 관계 |
|---|---|---|---|
| `treatGraph` | native. `Compound`·`Disease` 전부 + `TREATS` | 1,667 | 755 |
| `narrowGraph` | Cypher. 질병 3개에 붙은 `TREATS` 만 | 112 | 112 |
| `sameClassGraph` | Cypher. 원본에 없는 '같은 약효분류' 관계 | 621 | 3,156 |

### 🖐️ 함께 따라하기: 전철 그래프에 조건 걸어 담기

이번에는 **전철 그래프의 역-노선 축**에 조건을 겁니다.

`2호선`·`3호선`·`4호선` **세 노선과 그 노선이 지나는 역만** 담아 **`narrowLineGraph`** 라는 이름으로 Cypher 투영을 만들고, 노드 수와 관계 수를 출력하세요.

**확인 기준**: 노드 144개(역 141 + 노선 3), 관계 146건이 나오면 맞습니다. 2절에서 native 로 만든 `lineGraph`(노드 684 / 관계 802)와 견주어 보세요.

> **역이 141개인데 관계는 146건입니다.** 역 하나가 노선 하나에만 속한다면 두 수가 같아야 할 텐데 5건이 더 많죠. **환승역이 두 노선에 걸쳐 있기** 때문입니다. 이 세 노선 사이에서 갈아탈 수 있는 역이 5개이고, 그 역들은 `ON_LINE` 을 두 개씩 내보냅니다. 두 수가 어긋난다고 잘못 담긴 것이 아닙니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (s:Station)-[:ON_LINE]->(l:Line) 로 역-노선 짝을 찾는다
# 2) WHERE l.name IN ['2호선', '3호선', '4호선'] 로 세 노선만 남긴다
# 3) RETURN gds.graph.project('narrowLineGraph', s, l, {...}) AS g 로 담는다
#    설정 맵에는 sourceNodeLabels: labels(s), targetNodeLabels: labels(l) 를 넣는다
# 4) 돌려받은 맵에서 nodeCount 와 relationshipCount 만 골라 출력한다

### ✅ 바로 확인 퀴즈

**1.** "치료제가 붙은 질병만 담은 투영"을 native `gds.graph.project('이름', ['Disease'], ...)` 로 만들 수 있나요?

<details><summary>정답 보기</summary>

만들 수 없습니다. 레이블을 담으면 **그 레이블의 노드가 전부** 들어옵니다. 조건을 걸려면 `MATCH`·`WHERE` 로 고르는 **Cypher 투영**을 써야 합니다.

</details>

**2.** 예전 자료에서 본 `CALL gds.graph.project.cypher('이름', 노드쿼리, 관계쿼리)` 를 그대로 써도 될까요?

<details><summary>정답 보기</summary>

쓰지 마세요. 그 **프로시저**는 deprecated 라 언제 사라질지 모릅니다. 지금은 `RETURN` 자리에 쓰는 **집계 함수** `gds.graph.project(이름, 출발노드, 도착노드, 설정)` 가 그 자리를 대신합니다.

</details>

---
# 6. 차수(degree): 투영에 담긴 선을 세어 본다

투영을 만들고 관리하는 법까지 왔습니다. 이제 그 위에서 **계산을 한 번 돌려 봅니다.** 가장 단순한 것부터 시작합니다.

## 6-1. 차수를 재는 법

### 차수란: 방향이 있으면 한 가지가 아니다
**차수(degree)** 는 한 노드에 붙은 **선의 개수**입니다. "이 노드가 얼마나 중요한가"를 수 하나로 매기는 값을 **중심성(centrality)** 이라 하는데, 차수는 그 가장 단순한 형태입니다.

그런데 관계에 **방향이 있으면** "붙은 선"이 한 가지가 아닙니다. 셋으로 갈립니다.

| 이름 | 세는 것 | `(약)-[:BINDS]->(유전자)` 에서 |
|---|---|---|
| **외차수**(out-degree) | 그 노드에서 **나가는** 선 | 그 약이 붙는 표적의 개수 |
| **내차수**(in-degree) | 그 노드로 **들어오는** 선 | 그 표적에 붙는 약의 개수 |
| **전체 차수** | 둘을 합친 것 | 양쪽 다 |

> **`gds.degree.stream` 은 아무 설정 없이 부르면 외차수(out-degree)를 셉니다.** 이름이 그냥 `degree` 라 "붙은 선 전부"처럼 보이지만 아닙니다. **나가는 선만** 셉니다. 이 절은 그 기본값이 무엇을 세는지 눈으로 확인하는 데서 시작합니다.

```cypher
CALL gds.degree.stream('투영이름') YIELD nodeId, score
```

`nodeId` 는 **노드를 가리키는 번호**입니다(Neo4j 가 노드마다 붙여 둔 내부 id 와 같은 값이라 `MATCH (n) WHERE id(n) = 번호` 로도 찾힙니다). 이름 같은 속성은 이 번호에 담기지 않으므로 사람이 읽을 이름을 보려면 **`gds.util.asNode(nodeId)`** 로 실제 노드를 꺼냅니다.

이번 절은 **약물-질병-유전자** 축으로 봅니다. `TREATS`(약이 병을 치료한다)와 `BINDS`(약이 그 유전자가 만드는 단백질에 결합한다) 두 가지를 **방향을 살린 채** 담습니다. 방향을 지우면 외차수와 내차수를 나눌 수 없기 때문입니다.

In [ ]:
# 관계 두 종류를 방향 그대로 담는다(orientation 을 주지 않으면 원본 방향이 남는다)
res = run_cypher("""
    CALL gds.graph.project('targetGraph', ['Compound', 'Disease', 'Gene'],
                           ['TREATS', 'BINDS'])
    // 타입 둘을 함께 담아 둔다. 6-3 에서 이 투영 하나로 타입만 갈아 끼워 가며 묻는다
    YIELD nodeCount, relationshipCount
    RETURN nodeCount, relationshipCount
""")[0]
print(res)

In [ ]:
# 설정을 주지 않았으므로 이 셀이 세는 것은 '나가는 선'(외차수)이다
# score 는 실수로 오므로 개수로 읽으려면 int 로 바꾼다
for row in run_cypher("""
    CALL gds.degree.stream('targetGraph')   // stream 은 결과를 돌려주기만 하고 그래프에 쓰지 않는다
    YIELD nodeId, score
    RETURN gds.util.asNode(nodeId).name AS name,
           labels(gds.util.asNode(nodeId))[0] AS kind, score
    ORDER BY score DESC, name LIMIT 3   // 동점이 흔하다. name 을 두 번째 기준으로 둬야 순서가 고정된다
"""):
    print(f"  {row['name']:14} {row['kind']:10} 외차수 {int(row['score'])}")

1위가 `Sunitinib`(134)입니다. 이 약에서 **나가는** 선을 세어 보면 `BINDS` 132개 + `TREATS` 2개 = **134** 로 딱 맞습니다.

그런데 순위표가 이상합니다. 상위 세 자리가 전부 약물입니다. 유전자가 13,113개나 되는데 하나도 안 보이죠. 레이블별로 세어 보겠습니다.

In [ ]:
# 레이블마다 '가장 큰 외차수'와 '외차수가 0 인 노드 수'를 센다
for row in run_cypher("""
    CALL gds.degree.stream('targetGraph')
    YIELD nodeId, score
    WITH labels(gds.util.asNode(nodeId))[0] AS kind, score   // 레이블만 남기고 번호는 버린다
    // 집계 함수가 아닌 컬럼(kind)이 자동으로 묶는 기준이 된다. GROUP BY 를 따로 쓰지 않는다
    RETURN kind, count(*) AS 노드수, max(score) AS 최대,
           // 조건에 맞는 행만 세는 방법이다. CASE 로 1 과 0 을 만들어 그것을 더한다
           sum(CASE WHEN score = 0 THEN 1 ELSE 0 END) AS 차수0
    ORDER BY 최대 DESC
"""):
    print(f"  {row['kind']:10} 노드 {row['노드수']:>6,}  최대 {int(row['최대']):>4}  "
          f"외차수 0 인 노드 {row['차수0']:>6,}")

**유전자와 질병이 전부 0 점입니다.**

| 레이블 | 노드 수 | 최대 외차수 | 외차수 0 |
|---|---|---|---|
| `Compound` 약물 | 1,531 | 134 | 137 |
| `Gene` 유전자 | 13,113 | **0** | **13,113 전부** |
| `Disease` 질병 | 136 | **0** | **136 전부** |

노드 14,780개 중 13,386개가 0 이니 **90%가 넘습니다.** 계산이 틀린 것이 아닙니다. 이 투영의 화살표는 `TREATS`(약 → 병)도 `BINDS`(약 → 유전자)도 **약에서만 나갑니다.** 유전자와 질병은 받기만 하니 나가는 선이 0 개인 것이 맞습니다. `CYP3A4` 만 봐도 나가는 선 0 개, 들어오는 선 516 개입니다.

> **이 절에서 가져갈 것**: `gds.degree` 를 설정 없이 부르면 "붙은 선의 개수"가 아니라 **외차수**가 나옵니다. 방향이 살아 있는 투영에서 이것을 모르고 쓰면 답의 대부분이 0 이 되는데도 **에러가 나지 않습니다.** 순위표만 보고 "유전자는 안 중요하구나"라고 읽으면 그대로 틀립니다.

지금 센 값에는 전제가 **둘** 깔려 있습니다. **나가는 선만** 세었고, `TREATS` 와 `BINDS` 를 **구별하지 않고** 합쳐 세었습니다. 둘 다 바꿀 수 있습니다. 6-2 에서 방향을, 6-3 에서 관계 종류를 고릅니다.

> **무방향으로 담은 투영에서는 어떻게 되나요?** 기본값이 외차수라는 것은 **그대로**입니다. 다만 무방향 투영은 관계를 **양방향 두 벌로** 저장하므로, 나가는 선의 개수가 곧 **붙은 선 전체의 개수**가 됩니다. 그래서 설정 없이 불러도 결과적으로 전체 차수가 나옵니다. 3절에서 만든 `treatUndirected` 를 그냥 부르면 1위가 `hypertension`(68)인데, 이 68 은 나가는 선이 아니라 **고혈압에 붙은 치료 관계 전부**입니다(그 투영은 4절에서 내렸습니다. 6-2 에서 다시 만드는 법을 적어 둡니다).

## 6-2. 방향 고르기: `orientation`

`gds.degree` 는 기본으로 **나가는 선만** 셉니다. 두 번째 인자에 `orientation` 을 주면 바꿀 수 있습니다.

| `orientation` | 세는 것 | 부르는 이름 |
|---|---|---|
| `'NATURAL'`(기본) | 이 노드에서 **나가는** 선 | 외차수(out-degree) |
| `'REVERSE'` | 이 노드로 **들어오는** 선 | 내차수(in-degree) |
| `'UNDIRECTED'` | **양쪽 다** | 전체 차수 |

In [ ]:
# 같은 투영에 방향만 바꿔 가며 세 번 돌린다. 쿼리에서 바뀌는 곳은 orientation 한 군데뿐이다
for setting in ["NATURAL", "REVERSE", "UNDIRECTED"]:
    rows = run_cypher("""
        // 담을 때 준 orientation 이 아니다. 담긴 것은 그대로 두고 셀 때만 방향을 바꾼다
        CALL gds.degree.stream('targetGraph', {orientation: $orient})
        YIELD nodeId, score
        RETURN gds.util.asNode(nodeId).name AS name,
               labels(gds.util.asNode(nodeId))[0] AS kind, score
        ORDER BY score DESC, name LIMIT 1
    """, orient=setting)
    total = run_cypher("""
        CALL gds.degree.stream('targetGraph', {orientation: $orient})
        YIELD score
        RETURN sum(score) AS total   // 1위만 보지 않는다. 방향이 바뀌었는지는 합계로 검산한다
    """, orient=setting)
    print(f"{setting:12} 합계 {int(total[0]['total']):>7,}  1위 {rows[0]['name']:12} "
          f"{rows[0]['kind']:10} {int(rows[0]['score'])}")

**1위가 갈립니다.**

| 설정 | 합계 | 1위 | 이 순위가 답하는 질문 |
|---|---|---|---|
| `NATURAL` | 12,326 | `Sunitinib` 134 | 선을 **많이 내보내는** 것은? |
| `REVERSE` | 12,326 | `CYP3A4` 516 | 선을 **많이 받는** 것은? |
| `UNDIRECTED` | 24,652 | `CYP3A4` 516 | 선이 **가장 많이 붙은** 것은? |

합계를 보면 검산이 됩니다. 나가는 선을 다 세든 들어오는 선을 다 세든 **관계 하나를 한 번씩** 세는 것이라 둘 다 관계 수 `12,326` 입니다. 양쪽을 다 세면 관계 하나가 두 번 세어져 **2배**가 됩니다. 3절에서 무방향 투영의 관계 수가 2배였던 것과 같은 이야기입니다.

### 무방향으로 담은 투영에 `orientation` 을 주면
3절에서 만든 `treatUndirected` 는 `orientation: 'UNDIRECTED'` 로 담아 관계를 **양방향 두 벌**로 저장했습니다(원본 755건 → 투영 1,510건). 거기에 계산 설정을 주면 두 가지 일이 벌어집니다.

> 아래 표의 값은 **미리 재 둔 것**입니다. `treatUndirected` 는 4절에서 내렸기 때문입니다. 직접 확인하려면 한 줄로 다시 만들면 됩니다. `run_cypher("CALL gds.graph.project('treatUndirected', ['Compound','Disease'], {TREATS:{orientation:'UNDIRECTED'}}) YIELD graphName RETURN graphName")`

| | 방향을 살려 담은 투영(`TREATS` 만) | `treatUndirected`(무방향) |
|---|---|---|
| `NATURAL`(기본) | 나가는 선. 1위 `Methotrexate` 19 | 붙은 선 전부. 1위 `hypertension` 68 |
| `REVERSE` | 들어오는 선. 1위 `hypertension` 68 | **위와 같음.** 1위 `hypertension` 68 |
| `UNDIRECTED` | 양쪽. 합계가 2배 | ⚠️ **한 번 더 2배.** `hypertension` 이 **136** |

**첫째, `REVERSE` 가 아무 일도 하지 않습니다.** 나가는 선과 들어오는 선이 이미 같은 개수라 되돌릴 방향이 없습니다. 방향을 나눠 보고 싶다면 **투영을 만들 때** 살려 두는 수밖에 없습니다.

**둘째, `UNDIRECTED` 를 주면 답이 틀립니다.** 이미 두 벌인 관계를 GDS 가 또 양쪽으로 세어 차수가 한 번 더 2배가 됩니다. 고혈압에 붙은 치료 관계는 68건인데 **136** 으로 나옵니다. 합계도 1,510 에서 3,020 으로 부풀죠. 에러는 나지 않습니다.

> **무방향으로 담은 투영에는 `orientation` 을 주지 마세요.** 기본값 그대로가 맞습니다. `orientation` 은 **방향을 살려 담은 투영**에서만 뜻이 있습니다.

## 6-3. 관계 고르기: `relationshipTypes`

`relationshipTypes` 는 **담긴 관계 중 일부만** 골라 계산합니다. 노드는 그대로 두고 관계만 고르는 것이라, 투영을 새로 만들지 않고도 다른 질문을 던질 수 있습니다.

```cypher
CALL gds.degree.stream('투영이름', {relationshipTypes: ['BINDS'], orientation: 'REVERSE'})
```

이 두 설정을 함께 바꾸면 같은 투영 하나로 네 가지를 물을 수 있습니다.

In [ ]:
# 관계 타입과 방향을 짝지어 네 가지로 물어본다
for rel_type, orient, question in [
        ("BINDS", "NATURAL", "표적이 가장 많은 약"),
        ("BINDS", "REVERSE", "약이 가장 많이 붙는 표적"),
        ("TREATS", "NATURAL", "치료하는 병이 가장 많은 약"),
        ("TREATS", "REVERSE", "치료제가 가장 많은 병")]:
    rows = run_cypher("""
        CALL gds.degree.stream('targetGraph',
                               // relationshipTypes 는 리스트다. 둘 이상 넣으면 합쳐서 센다
                               {relationshipTypes: [$rel], orientation: $orient})
        YIELD nodeId, score
        RETURN gds.util.asNode(nodeId).name AS name,
               labels(gds.util.asNode(nodeId))[0] AS kind, score
        ORDER BY score DESC, name LIMIT 1
    """, rel=rel_type, orient=orient)
    # 1위의 레이블까지 찍는다. 방향을 바꾸면 답의 '종류'부터 달라지는 것이 이 절의 요점이다
    print(f"{question:18} {rows[0]['name']:14} {rows[0]['kind']:10} "
          f"{int(rows[0]['score'])}")

**투영은 하나인데 답이 넷입니다.**

| 관계 | 방향 | 1위 | 레이블 | 읽는 법 |
|---|---|---|---|---|
| `BINDS` | 나가는 | `Sunitinib` 132 | `Compound` | 가장 많은 표적에 붙는 **약** |
| `BINDS` | 들어오는 | `CYP3A4` 516 | `Gene` | 가장 많은 약이 붙는 **표적** |
| `TREATS` | 나가는 | `Methotrexate` 19 | `Compound` | 가장 많은 병에 쓰이는 **약** |
| `TREATS` | 들어오는 | `hypertension` 68 | `Disease` | 치료제가 가장 많은 **병** |

**레이블을 함께 보면 방향의 뜻이 분명해집니다.** 같은 `BINDS` 인데 나가는 쪽 1위는 `Compound`, 들어오는 쪽 1위는 `Gene` 입니다. 방향을 바꾸는 순간 답의 **종류부터** 달라집니다.

맨 아래 `68` 은 31일차에 Cypher 로 세었던 "고혈압에 쓰는 약 68개" 그 값입니다. 같은 답을 이번에는 **그래프 전체에 점수를 매겨 순위를 내면서** 얻었습니다. Cypher 는 고혈압 하나를 지목해 세었지만, 여기서는 질병 전부의 순위 중 1위로 나온 것입니다.

> **`relationshipTypes` 는 결과를 거르는 것이 아니라 그래프를 줄입니다.** 안 고른 관계는 계산에 아예 들어오지 않습니다. 그래서 남은 관계로 닿지 않는 노드는 차수가 0 이 됩니다.

### 🖐️ 함께 따라하기: 전철 그래프에서 방향 나눠 세기

이번에는 **전철 그래프의 역-노선 축**에서 방향을 나눠 셉니다.

2절에서 만든 `lineGraph` 는 `orientation` 을 주지 않고 담아 **방향이 그대로 남아** 있습니다. **새로 만들지 말고** 그 투영에 `orientation` 을 `NATURAL` 과 `REVERSE` 로 바꿔 가며 1위를 각각 출력하세요. 3절에서는 **담을 때** `orientation` 을 주어 방향을 지웠지만, 이 투영은 방향이 남아 있으니 6-2 에서 배운 대로 **차수를 셀 때** `orientation` 을 줍니다.

**2절 따라하기의 `lineGraph` 를 먼저 완성해야 이 셀이 돕니다.** 비워 두고 왔거나 **다른 이름으로 만들었다면** 2절로 돌아가 `lineGraph` 라는 이름으로 다시 실행하세요. `Graph with name 'lineGraph' does not exist` 가 나오면 그 뜻입니다.

**확인 기준**: 나가는 쪽 1위는 `김포공항`(`Station`, 5), 들어오는 쪽 1위는 `1호선`(`Line`, 102)입니다. 앞은 **가장 많은 노선이 지나는 역**이니 곧 환승역이고, 뒤는 **가장 많은 역을 거느린 노선**입니다. 레이블이 `Station` 에서 `Line` 으로 바뀌는 것이 곧 답의 종류가 달라졌다는 뜻입니다.

> **나가는 쪽은 1위가 동점입니다.** 노선 5개가 지나는 역은 `김포공항` 과 `서울역` 둘뿐이고, 이 둘의 점수가 5 로 같습니다. 그래서 `ORDER BY score DESC` 만 쓰면 어느 쪽이 먼저 나올지 정해지지 않습니다. `ORDER BY score DESC, name` 처럼 **이름을 두 번째 기준**으로 둬야 `김포공항` 으로 고정됩니다. `서울역` 이 나왔다면 답이 틀린 것이 아니라 이 정렬 기준이 빠진 것입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 2절에서 만든 lineGraph 를 그대로 쓴다. 새로 투영하지 않는다
# 2) NATURAL 과 REVERSE 로 각각 gds.degree.stream 을 돌린다
# 3) 각 1위의 이름·레이블(labels(...)[0])·차수를 출력한다
#    정렬은 ORDER BY score DESC, name 으로 한다. 나가는 쪽은 1위가 동점이라
#    이름을 두 번째 기준으로 두지 않으면 실행할 때마다 다른 역이 나올 수 있다

### ✅ 바로 확인 퀴즈

**1.** `gds.degree.stream` 을 설정 없이 부르면 무엇을 세나요?

<details><summary>정답 보기</summary>

그 노드에서 **나가는** 선만 셉니다(`orientation: 'NATURAL'` 이 기본). 들어오는 선까지 보려면 `'REVERSE'` 나 `'UNDIRECTED'` 를 줘야 합니다.

</details>

**2.** `NATURAL` 과 `REVERSE` 의 차수 합계가 둘 다 12,326 으로 같은 이유는?

<details><summary>정답 보기</summary>

관계 하나에는 출발점과 도착점이 하나씩 있습니다. 나가는 선을 다 세든 들어오는 선을 다 세든 **관계 하나를 한 번씩** 세는 것이라 합계가 관계 수와 같습니다. `UNDIRECTED` 는 관계 하나를 양끝에서 한 번씩 세므로 2배가 됩니다.

</details>

**3.** 무방향으로 담은 투영에 `orientation: 'REVERSE'` 를 주면 어떻게 되나요? `'UNDIRECTED'` 를 주면요?

<details><summary>정답 보기</summary>

`'REVERSE'` 는 **아무것도 바꾸지 않습니다.** 무방향 투영은 관계를 이미 양방향 두 벌로 저장해 나가는 선과 들어오는 선의 개수가 같기 때문입니다. 방향을 나눠 보려면 **투영을 만들 때** 살려 두어야 합니다.

`'UNDIRECTED'` 는 **답을 틀리게 만듭니다.** 이미 두 벌인 관계를 또 양쪽으로 세어 차수가 한 번 더 2배가 됩니다(`hypertension` 68 → 136). 에러는 나지 않으니 더 위험합니다. **무방향 투영에는 `orientation` 을 주지 않는 것이 맞습니다.**

</details>

---
## 🧹 다 쓴 투영 내리기

이번 시간에 배운 것 중 **가장 자주 잊는 것**이 이것입니다. 투영은 노트북 커널이 아니라 **Neo4j 서버 메모리**에 있습니다. 그래서 노트북을 닫아도 사라지지 않고, 같은 데이터베이스에 붙은 **다른 사람에게도 보이며**, 쌓이면 서버 메모리를 채웁니다.

실무에서 남이 만든 투영을 함부로 지울 수 없으니, **자기가 만든 것은 자기가 내리는 것**이 규칙입니다. 마무리로 오늘 만든 것을 정리합니다.

In [ ]:
# 지금 무엇이 남아 있는지 먼저 본다. 이름과 메모리를 함께 보면 무엇을 내릴지 정해진다
for row in run_cypher("""
    CALL gds.graph.list()
    YIELD graphName, nodeCount, memoryUsage
    RETURN graphName, nodeCount, memoryUsage ORDER BY graphName
"""):
    print(f"  {row['graphName']:18} 노드 {row['nodeCount']:>6,}  {row['memoryUsage']}")

In [ ]:
# 시연에서 만든 것만 내린다(따라하기 투영은 남긴다). false 를 주었으니 이미 없는 이름이 있어도 그냥 넘어간다
# 남의 투영까지 지우지 않으려고 이름을 하나씩 적는다(카탈로그 전체를 지우지 않는다)
for _name in ["treatGraph", "treatUndirected", "mixedGraph",
              "narrowGraph", "sameClassGraph", "targetGraph"]:
    run_cypher("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName",
               name=_name)
print("남은 투영:", [r["graphName"] for r in
      run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName")])

> **왜 이름을 하나씩 적나요?** `gds.graph.list()` 가 준 것을 전부 지우면 **남이 쓰는 중인 투영까지** 함께 내려갑니다. 투영은 세션이 아니라 서버에 있어서, 옆자리 동료가 30분 걸려 만든 것이 내 한 줄에 사라질 수 있습니다. 그 사람은 다음 알고리즘 호출이 실패할 때야 알게 됩니다. **지울 것을 이름으로 적는 습관**이 그래서 필요합니다.

---
## 이번 강의 정리

| 개념 | 핵심 |
|---|---|
| GDS | 그래프 **전역 계산**(중심성·경로 등)을 전담하는 분석 엔진. 플러그인으로 설치 |
| `gds.version()` | 설치·동작 확인 |
| 투영(projection) | 분석용 **메모리 사본**. `gds.graph.project(이름, 레이블, 관계)` |
| 레이블 리스트 | 노드 종류가 여럿이면 `['A', 'B']`. **양쪽 끝이 다 있어야** 관계가 담긴다 |
| 관계 0건 | 에러가 아니라 **레이블을 빠뜨렸다는 신호**. 만든 뒤 반드시 확인 |
| `orientation`(담을 때) | `'UNDIRECTED'` 로 방향을 지운다. 관계가 **2배**로 담긴다 |
| 검산 | 투영의 관계 수를 **Cypher 로 센 건수와 대조**한다(무방향이면 2배) |
| 목록·삭제 | `gds.graph.list()` · `gds.graph.drop(이름)` · `drop(이름, false)`(없어도 통과) |
| 메모리 | `gds.graph.list()` 의 `memoryUsage` · `gds.graph.project.estimate` 로 만들기 전에 가늠 |
| `YIELD` | 쓸 컬럼만 골라 받는다. 안 그러면 버전에 따라 경고가 뜬다 |
| Cypher 투영 | `RETURN gds.graph.project(이름, 출발, 도착, 설정)`. 조건을 걸거나 **원본에 없는 관계**를 만든다 |
| 차수(degree) | 노드에 붙은 선의 개수. `gds.degree.stream` 은 설정 없이 부르면 **외차수**(나가는 선)를 센다 |
| `orientation`(셀 때) | `'REVERSE'` 는 내차수, `'UNDIRECTED'` 는 전체 차수. **무방향으로 담은 투영에는 주지 않는다** |
| `relationshipTypes` | 담긴 관계 중 **일부만** 골라 계산한다. 안 고른 관계는 계산에 아예 들어오지 않는다 |
| 시점 사본 | 투영 이후 원본 변경은 **반영 안 됨**. 최신은 drop 후 다시 project |

> **남은 투영은 어떻게 하나요?** 위 정리 셀은 시연용만 내렸으므로 따라하기에서 만든 `lineGraph`·`narrowLineGraph` 는 남아 있습니다. `gds.graph.list()` 로 세어 보면 그대로 있습니다. 지금 손으로 내려도 되고, 그냥 두어도 됩니다. **다음 실행의 초기화 셀이 맨 먼저 투영부터 전부 내리기** 때문입니다. 다만 실습이 끝나고 DB 를 한동안 켜 둘 거라면 메모리를 잡고 있으니 내려 두는 편이 낫습니다.

**오늘 얻은 관점 하나**: 투영에 무엇을 담을지 고르는 일은 설정이 아니라 **질문의 범위를 정하는 일**입니다. 다음 시간에는 같은 그래프를 두 가지로 투영해 놓고, **투영이 달라지면 답이 어떻게 달라지는지** 봅니다.

## ⏭️ 예고: 다음 시간

오늘 차수로 "선이 몇 개인가"를 세었습니다. 다음 시간에는 **선이 어디에서 왔는지**까지 보는 중심성으로 넘어갑니다. **PageRank**·개인화 PageRank·매개 중심성·근접 중심성으로 같은 그래프에 네 가지 잣대를 대 보고, **답이 서로 얼마나 다른지** 직접 세어 봅니다.